In [583]:
import warnings
warnings.filterwarnings('ignore')

In [584]:
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt

from rapidfuzz import fuzz, process
import ftfy

from zipfile import ZipFile
from urllib.request import urlopen
from io import BytesIO
import unicodedata

import joblib

In [585]:
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vQZC7cVru6ltLR2e8XN5jdJPfxfj42BAxUApe3Zq3_ENQjLtYntmAxD0pIHqEUJ4ZFLXlybKJdkLf2r/pub?output=csv'
candinfo = pd.read_csv(url)
candinfo.to_csv('../../2026_data/2026_midterms_candidateinfo.csv')
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any
0,AK-AL,Bill Hill,Nick Begich,False,True
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False
2,AL-02,Shomari Figures,Hampton Harris,True,False
3,AL-03,Lee McInnis,Mike Rogers,False,True
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True


In [586]:
for party in ['dem', 'rep']:
    candinfo[f'{party}_cand'] = candinfo[f'{party}_cand'].fillna('TBD').astype(str)
    candinfo[f'{party}_inc_any'] = candinfo[f'{party}_inc_any'].astype(bool)

In [587]:
dem_only_candinfo = candinfo[candinfo['rep_cand'] == 'Not Contested']
rep_only_candinfo = candinfo[candinfo['dem_cand'] == 'Not Contested']
candinfo = candinfo[(candinfo['dem_cand'] != 'Not Contested') &
    (candinfo['rep_cand'] != 'Not Contested')]

In [588]:
dem_only_candinfo.shape, rep_only_candinfo.shape

((11, 5), (1, 5))

In [589]:
candinfo['state_po'] = candinfo['cd'].map(lambda x: x[:2]).astype(str)
candinfo['district_number'] = candinfo['cd'].map(lambda x: 0 if x[3:] == 'AL' else int(x[3:]))
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4


In [590]:
candinfo.shape

(423, 7)

In [591]:
# Code snippet courtesy of hantoine: https://gist.github.com/hantoine/c4fc70b32c2d163f604a8dc2a050d5f6
def download_and_unzip(url, extract_to='.'):
    http_candinfoponse = urlopen(url)
    zipfile = ZipFile(BytesIO(http_candinfoponse.read()))
    zipfile.extractall(path=extract_to)

In [592]:
fec_webl_url = 'https://www.fec.gov/files/bulk-downloads/2026/webl26.zip'
fec_cn_url = 'https://www.fec.gov/files/bulk-downloads/2026/cn26.zip'
download_and_unzip(fec_webl_url, extract_to='../../2026_data/fec')
download_and_unzip(fec_cn_url, extract_to='../../2026_data/fec')

In [593]:
fec_webl_colnames = ["CAND_ID", "CAND_NAME", "CAND_ICI", "PTY_CD", "CAND_PTY_AFFILIATION", "TTL_RECEIPTS", "TRANS_FROM_AUTH", "TTL_DISB", "TRANS_TO_AUTH", "COH_BOP", "COH_COP", "CAND_CONTRIB", "CAND_LOANS", "OTHER_LOANS", "CAND_LOAN_REPAY", "OTHER_LOAN_REPAY", "DEBTS_OWED_BY", "TTL_INDIV_CONTRIB", "CAND_OFFICE_ST", "CAND_OFFICE_DISTRICT", "SPEC_ELECTION", "PRIM_ELECTION", "RUN_ELECTION", "GEN_ELECTION", "GEN_ELECTION_PRECENT", "OTHER_POL_CMTE_CONTRIB", "POL_PTY_CONTRIB", "CVG_END_DT", "INDIV_REFUNDS", "CMTE_REFUNDS"]

In [594]:
fec_cn_colnames = ["CAND_ID", "CAND_NAME", "CAND_PTY_AFFILIATION", "CAND_ELECTION_YR", "CAND_OFFICE_ST", "CAND_OFFICE", "CAND_OFFICE_DISTRICT", "CAND_ICI", "CAND_STATUS", "CAND_PCC", "CAND_ST1", "CAND_ST2", "CAND_CITY", "CAND_ST", "CAND_ZIP"]

In [595]:
webl = pd.read_table('../../2026_data/fec/webl26.txt', sep="|", names=fec_webl_colnames)
cn = pd.read_table('../../2026_data/fec/cn.txt', sep="|", names=fec_cn_colnames)
fec = pd.merge(left=webl, right=cn, on='CAND_ID', how='left')
fec = fec[[col for col in fec.columns.values if ('_y' not in col)]]
fec.columns = fec.columns.str.strip('_x')
fec.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,...,CMTE_REFUNDS,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
0,H2AK01158,"PELTOLA, MARY",C,1,DEM,152304.86,0.00,232791.29,0.0,83969.49,...,0.00,2026.0,H,C,C00812388,810 N STREET,SUITE 301,ANCHORAGE,AK,99501.0
1,H6AK01084,"SCHULTZ, MATTHEW DAMIAN",C,1,DEM,865433.30,0.00,394752.27,1075.0,0.00,...,0.00,2026.0,H,C,C00923714,PO BOX 240641,NaN,ANCHORAGE,AK,99524.0
2,H2AK01083,"BEGICH, NICHOLAS III",I,2,REP,5470446.24,1601616.78,2455618.51,0.0,104330.06,...,4382.86,2026.0,H,C,C00792341,PO BOX 671710,NaN,CHUGIAK,AK,99567.0
3,H6AK01092,"HILL, BILL",C,3,IND,1336291.63,7400.00,820405.67,0.0,0.00,...,0.00,2026.0,H,C,C00935437,PO BOX 220703,NaN,ANCHORAGE,AK,99522.0
4,H6AL01094,"JONES, CLYDE W MR. JR",O,1,DEM,85967.63,0.00,57937.52,0.0,0.00,...,0.00,2026.0,H,C,C00920918,11637 WENTWOOD CT,NaN,DAPHNE,AL,36526.0


In [596]:
fec.columns.values

array(['CAND_ID', 'CAND_NAME', 'CAND_ICI', 'PTY_CD',
       'CAND_PTY_AFFILIATION', 'TTL_RECEIPTS', 'TRANS_FROM_AUTH',
       'TTL_DISB', 'TRANS_TO_AUTH', 'COH_BOP', 'COH_COP', 'CAND_CONTRIB',
       'CAND_LOANS', 'OTHER_LOANS', 'CAND_LOAN_REPAY', 'OTHER_LOAN_REPAY',
       'DEBTS_OWED_BY', 'TTL_INDIV_CONTRIB', 'CAND_OFFICE_ST',
       'CAND_OFFICE_DISTRICT', 'SPEC_ELECTION', 'PRIM_ELECTION',
       'RUN_ELECTION', 'GEN_ELECTION', 'GEN_ELECTION_PRECENT',
       'OTHER_POL_CMTE_CONTRIB', 'POL_PTY_CONTRIB', 'CVG_END_DT',
       'INDIV_REFUNDS', 'CMTE_REFUNDS', 'CAND_ELECTION_YR', 'CAND_OFFICE',
       'CAND_STATUS', 'CAND_PCC', 'CAND_ST1', 'CAND_ST2', 'CAND_CITY',
       'CAND_ST', 'CAND_ZIP'], dtype=object)

In [597]:
fec = fec[fec['CAND_OFFICE'] == 'H']
fec['CAND_OFFICE_ST'] = fec['CAND_OFFICE_ST'].astype(str)
fec['CAND_OFFICE_DISTRICT'] = fec['CAND_OFFICE_DISTRICT'].astype(float)
fec['CAND_NAME'] = fec['CAND_NAME'].astype(str)
fec['CAND_PTY_AFFILIATION'] = fec['CAND_PTY_AFFILIATION'].replace({'DFL': 'DEM'})

In [598]:
fec.shape

(2500, 39)

In [599]:
def get_fuzzymatch_cand(state_po, district, party, candidate):
    error_return = 'no_match'
    if candidate == 'TBD':
        return error_return
    
    df = fec[(fec['CAND_OFFICE_ST'] == state_po) &
        (fec['CAND_OFFICE_DISTRICT'] == district) &
        (fec['CAND_PTY_AFFILIATION'].isin([party, 'IND', 'NON', 'OTH']))]

    # df['cand_name_lst'] = df['CAND_NAME'].str.split(',')
    # df['first_name_cand'] = df['cand_name_lst'].map(lambda x: x[1])
    # df['last_name_cand'] = df['cand_name_lst'].map(lambda x: x[0])
    # df['cand_name_ordered'] = df['first_name_cand'] + ' ' + df['last_name_cand']

    if df.shape[0] == 0:
        return error_return

    if '/' in candidate:
        cands = candidate.split(separator='/')
        matches, contribs = [], []
        for c in cands:
            fuzzymatch = process.extractOne(c, df['CAND_NAME'].values, scorer=fuzz.WRatio, score_cutoff=55)
            indiv_contrib = df[df['CAND_NAME'] == fuzzymatch]['TTL_INDIV_CONTRIB'].values[0]
            if fuzzymatch is not None:
                matches.append(fuzzymatch[0])
                contribs.append(indiv_contrib)
            else:
                matches.append(error_return)
                contribs.append(0)
        return matches

    else:
        fuzzymatch = process.extractOne(candidate, df['CAND_NAME'].values, scorer=fuzz.token_sort_ratio, score_cutoff=10)
        if fuzzymatch is not None:
            indiv_contrib = df[df['CAND_NAME'] == fuzzymatch[0]]['TTL_INDIV_CONTRIB'].values[0]
        else:
            indiv_contrib = 0
        return error_return if fuzzymatch is None else fuzzymatch[0]

In [600]:
candinfo['fec_dem_fuzzymatch'] = candinfo.apply(lambda x: get_fuzzymatch_cand(x['state_po'], x['district_number'], 'DEM', x['dem_cand']), axis=1)
candinfo['fec_rep_fuzzymatch'] = candinfo.apply(lambda x: get_fuzzymatch_cand(x['state_po'], x['district_number'], 'REP', x['rep_cand']), axis=1)
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III"
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR"
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2,"FIGURES, SHOMARI C.","HARRIS, HAMPTON"
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL"
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","BARNES, THOMAS GARY"


In [601]:
dem_corrections = {
    ("CA-24", "Salud Carbajal"):    "CARBAJAL, SALUD O.",
    ("GA-01", "Amanda Hollowell"):  "HOLLOWELL, AMANDA",
    ("GA-07", "Tony Kozycki"):      "KOZYCKI, ANTHONY LAWRENCE",
    ("IL-08", "Melissa Bean"):      "BEAN, MELISSA LUBURICH",
    ("IN-09", "Brad Meyer"):        "MEYER, BRADLEY ALLEN MR.",
    ("MT-02", "Brian Miller"):      "MILLER, BRIAN JAMES",
    ("NC-12", "Alma Adams"):        "ADAMS, ALMA SHEALEY",
    ("NJ-07", "Rebecca Bennett"):   "BENNETT, REBECCA",
    ("NJ-11", "Analilia Mejia"):    "MEJIA, ANALILIA",
    ("NJ-12", "Adam Hamawy"):       "HAMAWY, ADAM",
    ("NY-12", "Micah Lasher"):      "LASHER, MICAH CHARLES",
    ("OH-15", "Don Leonard"):       "LEONARD, DON RALPH",
    ("PA-01", "Bob Harvie"):        "HARVIE, ROBERT J",
    ("TN-09", "Justin Pearson"):    "PEARSON, JUSTIN J.",
    ("TX-10", "Caitlin Rourk"):     "ROURK, CAITLIN MCCLAY",
    ("VA-08", "Don Beyer"):         "BEYER, DONALD STERNOFF JR.",
    ("TN-07", "Joshua Sales"):      "no_match",
    ("KY-02", "Megan Wingfield"):   "no_match",
    ("OH-05", "Brian Shaver"):      "no_match",
    ("OH-06", "Elizabeth Kirtley"): "no_match",
    ("OK-03", "Suzie Byrd"):        "no_match",
    ("TX-25", "Dione Sims"):        "no_match",
    ("MA-09", "Bill Keating"):      "KEATING, WILLIAM R",
    ("IA-01", "Christina Bohannan"): "BOHANNAN, CHRISTINA",
    ("KY-02", "Megan Wingfield"):    "WINGFIELD, MEGAN",
    ("MS-02", "Bennie Thompson"):    "THOMPSON, BENNIE G.",
    ("OH-05", "Brian Shaver"):       "SHAVER, BRIAN ALAN MR.",
    ("OH-06", "Elizabeth Kirtley"):  "KIRTLEY, ELIZABETH ANN MRS.",
    ("VA-01", "Shannon Taylor"):     "TAYLOR, SHANNON LEIGH",
}

rep_corrections = {
    ("AL-04", "Robert Aderholt"):      "ADERHOLT, ROBERT B. REP.",
    ("FL-01", "Jimmy Patronis"):       "PATRONIS, JIMMY JR.",
    ("GA-12", "Rick Allen"):           "ALLEN, RICHARD W",
    ("KY-06", "Ralph Alvarado"):       "ALVARADO, RALPH A",
    ("PA-05", "Nicholas Manganaro"):   "MANGANARO, NICHOLAS WALN MORRIS",
    ("TX-09", "Alex Mealer"):          "MEALER, ALEXANDRA",
    ("TX-21", "Mark Teixeira"):        "TEIXEIRA, MARK CHARLES",
    ("TX-22", "Trever Nehls"):         "NEHLS, TREVER",
    ("CA-06", "Kevin Kiley"):          "no_match",
    ("IL-04", "Lupe Castillo"):        "no_match",
    ("NY-15", "Stylo Sapaskis"):       "no_match",
    ("CA-02", "Robin Littau"):         "no_match",
    ("IL-09", "John Elleson"):         "no_match",
    ("KS-03", "Chase LaPorte"):        "no_match",
    ("MD-05", "Chris Chaffee"):        "no_match",
    ("NY-01", "Nick LaLota"):          "no_match",
    ("NY-09", "Joel Anabilah-Azumah"): "no_match",
    ("PA-04", "Aurora Stuski"):        "no_match",
    ("TX-18", "Ronald Whitfield"):     "no_match",
    ("MI-04", "Bill Huizenga"):        "HUIZENGA, WILLIAM P",
    ("MO-06", "Chris Stigall"):                     "STIGALL, CHRIS",
    ("MI-13", "T.P. Nykoriak"):  "no_match",
    ("MO-03", "Bob Onder"):      "ONDER, ROBERT FOR JR.",
    ("MO-05", "Rick Brattin"):   "BRATTIN, RICHARD",
    ("VA-11", "Arthur Purves"):  "PURVES, ARTHUR GRAHAME MR.",
}

In [602]:
for (cd, cand) in dem_corrections.keys():
    mask = (
        (candinfo['cd'] == cd) &
        (candinfo['dem_cand'] == cand)
    )

    candinfo.loc[mask, 'fec_dem_fuzzymatch'] = dem_corrections[(cd, cand)]

for (cd, cand) in rep_corrections.keys():
    mask = (
        (candinfo['cd'] == cd) &
        (candinfo['rep_cand'] == cand)
    )

    candinfo.loc[mask, 'fec_rep_fuzzymatch'] = rep_corrections[(cd, cand)]

In [603]:
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III"
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR"
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2,"FIGURES, SHOMARI C.","HARRIS, HAMPTON"
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL"
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP."


In [604]:
def get_indiv_contribs(state_po, district_number, party, candidate):
    if candidate == 'TBD':
        return 0
    
    df = fec[
        (fec['CAND_OFFICE_ST'] == state_po) &
        (fec['CAND_OFFICE_DISTRICT'] == district_number) &
        (fec['CAND_PTY_AFFILIATION'].isin([party, 'IND', 'OTH', 'NON'])) &
        (fec['CAND_NAME'] == candidate)
    ]

    if df.shape[0] == 0:
        return 0

    return df['TTL_INDIV_CONTRIB'].values[0]

candinfo['dem_funds'] = candinfo.apply(lambda x: get_indiv_contribs(x['state_po'], x['district_number'], 'DEM', x['fec_dem_fuzzymatch']), axis=1)
candinfo['rep_funds'] = candinfo.apply(lambda x: get_indiv_contribs(x['state_po'], x['district_number'], 'REP', x['fec_rep_fuzzymatch']), axis=1)
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",1264611.09,2784023.76
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",83967.63,674069.54
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2,"FIGURES, SHOMARI C.","HARRIS, HAMPTON",490903.40,38740.30
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL",60787.05,1356433.59
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP.",15462.00,523823.01


In [605]:
candinfo['2p_funds'] = candinfo['dem_funds'] + candinfo['rep_funds']
candinfo['dem_funds_2p_pct'] = candinfo['dem_funds'] / candinfo['2p_funds'] * 100
candinfo['rep_funds_2p_pct'] = candinfo['rep_funds'] / candinfo['2p_funds'] * 100
for party in ['dem', 'rep']:
    candinfo[f'{party}_funds_2p_pct'] = candinfo[f'{party}_funds_2p_pct'].fillna(0) 
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",1264611.09,2784023.76,4048634.85,31.235494,68.764506
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",83967.63,674069.54,758037.17,11.076981,88.923019
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2,"FIGURES, SHOMARI C.","HARRIS, HAMPTON",490903.40,38740.30,529643.70,92.685592,7.314408
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL",60787.05,1356433.59,1417220.64,4.289173,95.710827
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP.",15462.00,523823.01,539285.01,2.867130,97.132870


In [606]:
## Wrangle PVI + demographic data for redistricted states
redist_dfs_2020 = []
redist_dfs_2024 = []

redist_states = ['CA', 'UT', 'TN', 'TX', 'LA', 'MO', 'FL', 'AL', 'NC', 'OH']
for st in redist_states:
    dat_2020 = pd.read_csv(f'../../2026_data/dra/{st}_2026_distdata_2020_elec.csv').iloc[1:]
    dat_2024 = pd.read_csv(f'../../2026_data/dra/{st}_2026_distdata_2024_elec.csv').iloc[1:]
    dat_2020['state'] = np.full(dat_2020.shape[0], fill_value=st)
    dat_2024['state'] = np.full(dat_2024.shape[0], fill_value=st)
    redist_dfs_2020.append(dat_2020[['state', 'Label', 'V_24_CVAP_Total', 'V_24_CVAP_White', 'V_24_CVAP_Hispanic', 'V_24_CVAP_BlackAlone', 'V_24_CVAP_AsianAlone',
                               'V_24_CVAP_NativeAlone', 'V_24_CVAP_PacificAlone', 'T_22_ACS_Total', 'E_20_PRES_Dem', 'E_20_PRES_Rep', 'X_22_2022_Education_Bach',
                               'X_22_2022_Education_Master', 'X_22_2022_Education_Prof', 'X_22_2022_Education_Doc']])
    redist_dfs_2024.append(dat_2024[['state', 'Label', 'V_24_CVAP_Total', 'V_24_CVAP_White', 'V_24_CVAP_Hispanic', 'V_24_CVAP_BlackAlone', 'V_24_CVAP_AsianAlone',
                               'V_24_CVAP_NativeAlone', 'V_24_CVAP_PacificAlone', 'T_22_ACS_Total', 'E_24_PRES_Dem', 'E_24_PRES_Rep', 'X_22_2022_Education_Bach',
                               'X_22_2022_Education_Master', 'X_22_2022_Education_Prof', 'X_22_2022_Education_Doc']])

redist_df_20 = pd.concat(redist_dfs_2020, axis=0)
redist_df_24 = pd.concat(redist_dfs_2024, axis=0)

In [607]:
rename_col_dict = {
    'state': 'state_po',
    'Label': 'district_number',
    'V_24_CVAP_Total': 'cvap',
    'V_24_CVAP_White': 'white',
    'V_24_CVAP_Hispanic': 'hispanic',
    'V_24_CVAP_BlackAlone': 'black',
    'V_24_CVAP_AsianAlone': 'asian',
    'V_24_CVAP_NativeAlone': 'natam',
    'V_24_CVAP_PacificAlone': 'pi',
    'X_22_2022_Education_Bach': 'bachelors',
    'X_22_2022_Education_Master': 'masters',
    'X_22_2022_Education_Prof': 'prof',
    'X_22_2022_Education_Doc': 'doc',
    'T_22_ACS_Total': 'tot_22'
}
redist_df_20 = redist_df_20.rename(rename_col_dict, axis=1)
redist_df_24 = redist_df_24.rename(rename_col_dict, axis=1)
redist_df_20 = redist_df_20.rename({'E_20_PRES_Dem': 'dem_20', 'E_20_PRES_Rep': 'rep_20'}, axis=1)
redist_df_24 = redist_df_24.rename({'E_24_PRES_Dem': 'dem_24', 'E_24_PRES_Rep': 'rep_24'}, axis=1)

In [608]:
redist_df_20.head()

,state_po,district_number,cvap,white,hispanic,black,asian,natam,pi,tot_22,dem_20,rep_20,bachelors,masters,prof,doc
1,CA,1,546258,386988,104698,9496,18419,6393,1303,765529,212964,147017,96660,35283,12280,5840
2,CA,2,570919,448939,60753,8980,20488,8308,1172,761273,263170,144491,130290,55148,22334,10839
3,CA,3,552614,362996,82509,32392,48632,1624,2356,761545,217623,163088,125518,49682,16192,7454
4,CA,4,538599,321434,127903,14405,48564,2218,1987,759036,214496,138292,99488,40294,13242,9755
5,CA,5,565163,355588,139806,14885,28229,4920,2355,767406,159852,215543,97117,32763,12573,6018


In [609]:
redist_df_24.head()

,state_po,district_number,cvap,white,hispanic,black,asian,natam,pi,tot_22,dem_24,rep_24,bachelors,masters,prof,doc
1,CA,1,546258,386988,104698,9496,18419,6393,1303,765529,186164,144424,96660,35283,12280,5840
2,CA,2,570919,448939,60753,8980,20488,8308,1172,761273,236524,140558,130290,55148,22334,10839
3,CA,3,552614,362996,82509,32392,48632,1624,2356,761545,195637,158302,125518,49682,16192,7454
4,CA,4,538599,321434,127903,14405,48564,2218,1987,759036,191758,141847,99488,40294,13242,9755
5,CA,5,565163,355588,139806,14885,28229,4920,2355,767406,139201,214114,97117,32763,12573,6018


In [610]:
redist_df_20['cvap_white_pct'] = redist_df_20['white'] / redist_df_20['cvap'] * 100
redist_df_20['cvap_hisp_pct'] = redist_df_20['hispanic'] / redist_df_20['cvap'] * 100
redist_df_20['cvap_natam_pct'] = redist_df_20['natam'] / redist_df_20['cvap'] * 100
redist_df_20['cvap_black_pct'] = redist_df_20['black'] / redist_df_20['cvap'] * 100
redist_df_20['aapi'] = redist_df_20['asian'] + redist_df_20['pi']
redist_df_20['cvap_aapi_pct'] = redist_df_20['aapi'] / redist_df_20['cvap'] * 100
redist_df_20['college'] = 100 * (redist_df_20['bachelors'] + redist_df_20['masters'] + redist_df_20['prof'] + redist_df_20['doc']) / redist_df_20['tot_22']
redist_df_20 = pd.merge(left=redist_df_20, right=redist_df_24[['state_po', 'district_number', 'dem_24', 'rep_24']], 
                        on=['state_po', 'district_number'], how='left')
redist_df_20.head()

,state_po,district_number,cvap,white,hispanic,black,asian,natam,pi,tot_22,...,doc,cvap_white_pct,cvap_hisp_pct,cvap_natam_pct,cvap_black_pct,aapi,cvap_aapi_pct,college,dem_24,rep_24
0,CA,1,546258,386988,104698,9496,18419,6393,1303,765529,...,5840,70.843448,19.166401,1.170326,1.738373,19722,3.610382,19.602523,186164,144424
1,CA,2,570919,448939,60753,8980,20488,8308,1172,761273,...,10839,78.634447,10.641264,1.455198,1.572903,21660,3.793883,28.716505,236524,140558
2,CA,3,552614,362996,82509,32392,48632,1624,2356,761545,...,7454,65.687080,14.930675,0.293876,5.861596,50988,9.226693,26.110867,195637,158302
3,CA,4,538599,321434,127903,14405,48564,2218,1987,759036,...,9755,59.679650,23.747352,0.411809,2.674532,50551,9.385647,21.445491,191758,141847
4,CA,5,565163,355588,139806,14885,28229,4920,2355,767406,...,6018,62.917778,24.737288,0.870545,2.633753,30584,5.411536,19.347125,139201,214114


In [611]:
# Source: Wikipedia, FEC
# Links:
# https://en.wikipedia.org/wiki/2024_United_States_presidential_election#Electoral_results
# https://www.fec.gov/resources/cms-content/documents/federalelections2020.pdf
# https://www.fec.gov/resources/cms-content/documents/federalelections2016.pdf#page=10
# https://www.fec.gov/resources/cms-content/documents/federalelections2012.pdf#page=11
dem_2pv_24 = 75017613 / (75017613 + 77302580) * 100
dem_2pv_20 = 81283501 / (81283501 + 74223975) * 100
dem_2pv_16 = 65853514 / (65853514 + 62984828) * 100
dem_2pv_12 = 65915795 / (65915795 + 60933504) * 100
dem_2pv_12, dem_2pv_16, dem_2pv_20, dem_2pv_24

(51.96386225200976, 51.113288930712876, 52.26983492420647, 49.24994613156773)

In [612]:
redist_df = redist_df_20.copy()
redist_df['2p_vote_20'] = redist_df['dem_20'] + redist_df['rep_20']
redist_df['2p_vote_24'] = redist_df['dem_24'] + redist_df['rep_24']
for yr in [20, 24]:
    redist_df[f'dem_2p_pct_{yr}'] = redist_df[f'dem_{yr}'] / redist_df[f'2p_vote_{yr}'] * 100
    redist_df[f'rep_2p_pct_{yr}'] = redist_df[f'rep_{yr}'] / redist_df[f'2p_vote_{yr}'] * 100

redist_df['lean_20'] = redist_df['dem_2p_pct_20'] - dem_2pv_20
redist_df['lean_24'] = redist_df['dem_2p_pct_24'] - dem_2pv_24
redist_df['pvi'] = 0.75 * redist_df['lean_24'] + 0.25 * redist_df['lean_20']
redist_df.head()

,state_po,district_number,cvap,white,hispanic,black,asian,natam,pi,tot_22,...,rep_24,2p_vote_20,2p_vote_24,dem_2p_pct_20,rep_2p_pct_20,dem_2p_pct_24,rep_2p_pct_24,lean_20,lean_24,pvi
0,CA,1,546258,386988,104698,9496,18419,6393,1303,765529,...,144424,359981,330588,59.159789,40.840211,56.312994,43.687006,6.889954,7.063048,7.019774
1,CA,2,570919,448939,60753,8980,20488,8308,1172,761273,...,140558,407661,377082,64.556089,35.443911,62.724818,37.275182,12.286255,13.474872,13.177718
2,CA,3,552614,362996,82509,32392,48632,1624,2356,761545,...,158302,380711,353939,57.162257,42.837743,55.274214,44.725786,4.892422,6.024268,5.741306
3,CA,4,538599,321434,127903,14405,48564,2218,1987,759036,...,141847,352788,333605,60.800254,39.199746,57.480553,42.519447,8.530419,8.230607,8.305560
4,CA,5,565163,355588,139806,14885,28229,4920,2355,767406,...,214114,375395,353315,42.582347,57.417653,39.398554,60.601446,-9.687488,-9.851392,-9.810416


In [613]:
# other_dists = pd.read_csv('../../transformed/all_2p_house_races_trainset.csv')
# other_dists = other_dists[other_dists['year'] == 2024]
# other_dists = other_dists[~(other_dists['state_po'].isin(redist_states))]
# other_dists = other_dists[['state', 'state_po', 'district', 'cvap_hisp_pct', 'cvap_aapi_pct', 'cvap_black_pct', 'cvap_natam_pct',
#                           'college']]
# other_dists.head()

In [614]:
other_dists = pd.read_csv('../../2026_data/demographics.csv')
other_dists['state_po'] = other_dists['district'].map(lambda x: x[:2])
other_dists = other_dists[~(other_dists['state_po'].isin(redist_states))]
other_dists = other_dists.drop(['Unnamed: 0'], axis=1)
other_dists.head()

,district,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,state_po
0,AK-00,64.463948,6.481397,6.051383,3.077471,12.640423,30.75,AK
1,DE-00,66.185281,3.007143,6.848324,21.107845,0.148276,34.46,DE
2,ND-00,87.593513,0.869295,3.388135,2.122269,3.724026,31.43,ND
3,SD-00,85.992821,0.931722,2.699753,1.494189,6.152199,30.37,SD
4,VT-00,93.220927,1.247612,2.220404,1.008821,0.121503,41.72,VT


In [615]:
pvi24 = pd.read_csv('../../transformed/pvi/past_pres_results_by24dist.csv')
pvi24['state_po'] = pvi24['district'].astype(str).map(lambda x: x[:2])
pvi24_all = pvi24.copy()
pvi24 = pvi24[~(pvi24['state_po'].isin(redist_states))]
pvi24['pvi'] = pvi24['lean_24'] * 0.75 + pvi24['lean_20'] * 0.25
pvi24['district'] = pvi24['district'].map(lambda x: x[:2] + '-00' if x[3:] == 'AL' else x)
pvi24.head()

,Unnamed: 0,district,incumbent,party,dem_24,rep_24,tot_24,dem_pct_24,rep_pct_24,dem_20,...,rep_16,tot_16,2p_tot_16,dem_2p_16,rep_2p_16,lean_16,lean_20,lean_24,state_po,pvi
0,2,AK-00,Nick Begich,(R),140026,184458,338177,41.41,54.54,153778,...,163387,309407,279841,41.614345,58.385655,-9.498944,-7.531684,-6.096509,AK,-6.455302
8,10,AR-01,Rick Crawford,(R),72183,196035,273446,26.40,71.69,80516,...,178526,274701,261236,31.661027,68.338973,-19.452262,-23.343455,-22.337882,AR,-22.589275
9,11,AR-02,French Hill,(R),131089,180710,319500,41.03,56.56,140254,...,166843,306119,287686,42.005172,57.994828,-9.108117,-9.010670,-7.207156,AR,-7.658034
10,12,AR-03,Steve Womack,(R),112073,189161,309672,36.19,61.08,110479,...,156678,259373,239154,34.486565,65.513435,-16.626724,-14.340126,-12.045315,AR,-12.619017
11,13,AR-04,Bruce Westerman,(R),81560,193335,280058,29.12,69.03,92683,...,182697,290299,277113,34.071299,65.928701,-17.041989,-20.391524,-19.580436,AR,-19.783208


In [616]:
other_dists = pd.merge(left=other_dists, right=pvi24[['district', 'lean_20', 'lean_24', 'pvi']], on='district', how='right')
other_dists.head()

,district,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,state_po,lean_20,lean_24,pvi
0,AK-00,64.463948,6.481397,6.051383,3.077471,12.640423,30.75,AK,-7.531684,-6.096509,-6.455302
1,AR-01,78.395768,0.458352,2.443255,15.943725,0.240072,17.57,AR,-23.343455,-22.337882,-22.589275
2,AR-02,72.754273,1.055418,3.446925,20.171494,0.211821,32.53,AR,-9.010670,-7.207156,-7.658034
3,AR-03,80.427735,2.733818,9.752721,2.673254,0.730994,31.07,AR,-14.340126,-12.045315,-12.619017
4,AR-04,72.999456,0.570913,4.195911,19.516478,0.322135,18.06,AR,-20.391524,-19.580436,-19.783208


In [617]:
other_dists['pvi'].isna().any()

np.False_

In [618]:
other_dists['district_number'] = other_dists['district'].map(lambda x: int(x[3:]))
other_dists.head(2)

,district,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,state_po,lean_20,lean_24,pvi,district_number
0,AK-00,64.463948,6.481397,6.051383,3.077471,12.640423,30.75,AK,-7.531684,-6.096509,-6.455302,0
1,AR-01,78.395768,0.458352,2.443255,15.943725,0.240072,17.57,AR,-23.343455,-22.337882,-22.589275,1


In [619]:
# Code snippet courtesy of rogerallen: https://gist.github.com/rogerallen/1583593
us_state_to_abbrev = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
    "American Samoa": "AS",
    "Guam": "GU",
    "Northern Mariana Islands": "MP",
    "Puerto Rico": "PR",
    "United States Minor Outlying Islands": "UM",
    "Virgin Islands, U.S.": "VI",
} 

abbrev_to_us_state = dict(map(reversed, us_state_to_abbrev.items()))

In [620]:
redist_df['state'] = redist_df['state_po'].map(abbrev_to_us_state)
other_dists['state'] = other_dists['state_po'].map(abbrev_to_us_state)
redist_df['district'] = redist_df['state_po'].astype(str) + '-' + redist_df['district_number'].map(lambda x: f'0{x}' if int(x) < 10 else str(x))
redist_df.head(2)

,state_po,district_number,cvap,white,hispanic,black,asian,natam,pi,tot_22,...,2p_vote_24,dem_2p_pct_20,rep_2p_pct_20,dem_2p_pct_24,rep_2p_pct_24,lean_20,lean_24,pvi,state,district
0,CA,1,546258,386988,104698,9496,18419,6393,1303,765529,...,330588,59.159789,40.840211,56.312994,43.687006,6.889954,7.063048,7.019774,California,CA-01
1,CA,2,570919,448939,60753,8980,20488,8308,1172,761273,...,377082,64.556089,35.443911,62.724818,37.275182,12.286255,13.474872,13.177718,California,CA-02


In [621]:
all_dists = pd.concat([
    redist_df[['state', 'state_po', 'district_number', 'district', 'cvap_white_pct', 'cvap_hisp_pct', 'cvap_aapi_pct', 'cvap_natam_pct',
              'cvap_black_pct', 'college', 'lean_20', 'lean_24', 'pvi']],
    other_dists[['state', 'state_po', 'district_number', 'district', 'cvap_white_pct', 'cvap_hisp_pct', 'cvap_aapi_pct', 'cvap_natam_pct',
              'cvap_black_pct', 'college', 'lean_20', 'lean_24', 'pvi']]
], axis=0)
all_dists = all_dists.sort_values(by=['district'])
all_dists.head()

,state,state_po,district_number,district,cvap_white_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_black_pct,college,lean_20,lean_24,pvi
0,Alaska,AK,0,AK-00,64.463948,6.051383,6.481397,12.640423,3.077471,30.750000,-7.531684,-6.096509,-6.455302
145,Alabama,AL,1,AL-01,69.156017,2.677114,1.299003,0.469439,24.385648,17.996871,-17.482017,-17.422182,-17.437141
146,Alabama,AL,2,AL-02,53.493739,2.691740,1.032138,0.183181,40.524512,16.170153,-6.640350,-6.482160,-6.521708
147,Alabama,AL,3,AL-03,74.925713,2.047228,0.853803,0.164703,19.752354,15.301623,-22.977818,-22.768849,-22.821092
148,Alabama,AL,4,AL-04,86.997080,3.646458,0.489492,0.174054,6.737716,13.622184,-33.634273,-32.744762,-32.967139


In [622]:
all_dists.shape

(435, 13)

In [623]:
all_dists['district_number'] = all_dists['district_number'].astype(int)
all_dists['state_po'] = all_dists['state_po'].astype(str)

In [624]:
candinfo = pd.merge(left=candinfo, right=all_dists, on=['state_po', 'district_number'], how='left')
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,...,district,cvap_white_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_black_pct,college,lean_20,lean_24,pvi
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",1264611.09,...,AK-00,64.463948,6.051383,6.481397,12.640423,3.077471,30.750000,-7.531684,-6.096509,-6.455302
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",83967.63,...,AL-01,69.156017,2.677114,1.299003,0.469439,24.385648,17.996871,-17.482017,-17.422182,-17.437141
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2,"FIGURES, SHOMARI C.","HARRIS, HAMPTON",490903.40,...,AL-02,53.493739,2.691740,1.032138,0.183181,40.524512,16.170153,-6.640350,-6.482160,-6.521708
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL",60787.05,...,AL-03,74.925713,2.047228,0.853803,0.164703,19.752354,15.301623,-22.977818,-22.768849,-22.821092
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP.",15462.00,...,AL-04,86.997080,3.646458,0.489492,0.174054,6.737716,13.622184,-33.634273,-32.744762,-32.967139


In [625]:
genbal = pd.read_csv('../../../snoutcounter-backend/averages/generic_ballot.csv')
gb_curr = genbal['net'].values[-1]
gb_curr

np.float64(-6.714078774876553)

In [626]:
candinfo['generic_ballot_avg'] = np.full(shape=candinfo.shape[0], fill_value=gb_curr)
candinfo.head(3)

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,...,cvap_white_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_black_pct,college,lean_20,lean_24,pvi,generic_ballot_avg
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",1264611.09,...,64.463948,6.051383,6.481397,12.640423,3.077471,30.750000,-7.531684,-6.096509,-6.455302,-6.714079
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",83967.63,...,69.156017,2.677114,1.299003,0.469439,24.385648,17.996871,-17.482017,-17.422182,-17.437141,-6.714079
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2,"FIGURES, SHOMARI C.","HARRIS, HAMPTON",490903.40,...,53.493739,2.691740,1.032138,0.183181,40.524512,16.170153,-6.640350,-6.482160,-6.521708,-6.714079


In [627]:
candinfo['dem_inc_dummy'] = candinfo['dem_inc_any'].map(lambda x: 0 if x == False else 1)
candinfo['rep_inc_dummy'] = candinfo['rep_inc_any'].map(lambda x: 0 if x == False else 1)
candinfo.head(3)

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,...,cvap_aapi_pct,cvap_natam_pct,cvap_black_pct,college,lean_20,lean_24,pvi,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",1264611.09,...,6.481397,12.640423,3.077471,30.750000,-7.531684,-6.096509,-6.455302,-6.714079,0,1
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",83967.63,...,1.299003,0.469439,24.385648,17.996871,-17.482017,-17.422182,-17.437141,-6.714079,0,0
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2,"FIGURES, SHOMARI C.","HARRIS, HAMPTON",490903.40,...,1.032138,0.183181,40.524512,16.170153,-6.640350,-6.482160,-6.521708,-6.714079,1,0


In [628]:
poll_avg = pd.read_csv('transformed/house_polling_averages.csv')
poll_avg.head()

,state,seat_number,candidate_name,party,avg,std,lower_ci,upper_ci,effn
0,AK,1,Bill Hill,IND,47.981778,0.447341,47.10499,48.858566,1.703864
1,AK,1,Nick Begich,REP,51.981778,0.447341,51.10499,52.858566,1.703864
2,AK,1,Don't know,NONE,22.000000,0.000000,22.00000,22.000000,1.003864
3,AR,2,Don't know,NONE,NaN,NaN,NaN,NaN,0.700000
4,AR,2,French Hill,REP,NaN,NaN,NaN,NaN,0.700000


In [629]:
poll_avg['party'].value_counts()

party
DEM     47
REP     46
NONE    41
IND      2
Name: count, dtype: int64

In [630]:
def get_polling_average(state, seat_number, party, candidate_name):
    df = poll_avg[
        (poll_avg['state'] == state) &
        (poll_avg['seat_number'] == seat_number) &
        (poll_avg['party'].isin([party, 'IND']))
    ]
    if df.shape[0] == 0 or process.extractOne(candidate_name, df['candidate_name'], scorer=fuzz.token_sort_ratio, score_cutoff=30) is None:
        # lax score cutoff for fuzzy string matching - make sure to check manually when appropriate
        return 0, 0
    avg = df['avg'].values[0]
    effn = df['effn'].values[0]
    return (avg, effn)

In [631]:
for party in ['dem', 'rep']:
    candinfo[[f'{party}_poll_avg', f'{party}_effn']] = candinfo.apply(lambda x: get_polling_average(x['state_po'], x['district_number'],
                                                                                                   party.upper(), x[f'{party}_cand']), axis=1,
                                                                     result_type='expand')
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,...,lean_20,lean_24,pvi,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",1264611.09,...,-7.531684,-6.096509,-6.455302,-6.714079,0,1,0.0,0.0,0.0,0.0
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",83967.63,...,-17.482017,-17.422182,-17.437141,-6.714079,0,0,0.0,0.0,0.0,0.0
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2,"FIGURES, SHOMARI C.","HARRIS, HAMPTON",490903.40,...,-6.640350,-6.482160,-6.521708,-6.714079,1,0,0.0,0.0,0.0,0.0
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL",60787.05,...,-22.977818,-22.768849,-22.821092,-6.714079,0,1,0.0,0.0,0.0,0.0
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP.",15462.00,...,-33.634273,-32.744762,-32.967139,-6.714079,0,1,0.0,0.0,0.0,0.0


In [632]:
# random intercepts for candidates - want to make distinct 'TBD' candidates for each seat
def distinct_tbd(cand, cd):
    if cand == 'TBD':
        return cand + cd
    else:
        return cand
candinfo['dem_cand'] = candinfo.apply(lambda x: distinct_tbd(x['dem_cand'], x['cd']), axis=1)
candinfo['rep_cand'] = candinfo.apply(lambda x: distinct_tbd(x['rep_cand'], x['cd']), axis=1)
candinfo.head(2)

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,...,lean_20,lean_24,pvi,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",1264611.09,...,-7.531684,-6.096509,-6.455302,-6.714079,0,1,0.0,0.0,0.0,0.0
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",83967.63,...,-17.482017,-17.422182,-17.437141,-6.714079,0,0,0.0,0.0,0.0,0.0


In [633]:
state_to_census_region = {
    'Connecticut': 'New England',
    'Maine': 'New England',
    'Massachusetts': 'New England',
    'New Hampshire': 'New England',
    'Rhode Island': 'New England',
    'Vermont': 'New England',
    'New Jersey': 'Mid-Atlantic',
    'New York': 'Mid-Atlantic',
    'Pennsylvania': 'Mid-Atlantic',
    'Illinois': 'East North Central',
    'Indiana': 'East North Central',
    'Michigan': 'East North Central',
    'Ohio': 'East North Central',
    'Wisconsin': 'East North Central',
    'Iowa': 'West North Central',
    'Kansas': 'West North Central',
    'Minnesota': 'West North Central',
    'Missouri': 'West North Central',
    'Nebraska': 'West North Central',
    'North Dakota': 'West North Central',
    'South Dakota': 'West North Central',
    'Delaware': 'South Atlantic',
    'Maryland': 'South Atlantic',
    'Florida': 'South Atlantic',
    'Georgia': 'South Atlantic',
    'North Carolina': 'South Atlantic',
    'South Carolina': 'South Atlantic',
    'Virginia': 'South Atlantic',
    'West Virginia': 'South Atlantic',
    'Alabama': 'East South Central',
    'Kentucky': 'East South Central',
    'Mississippi': 'East South Central',
    'Tennessee': 'East South Central',
    'Arkansas': 'West South Central',
    'Louisiana': 'West South Central',
    'Oklahoma': 'West South Central',
    'Texas': 'West South Central',
    'Arizona': 'Mountain',
    'Colorado': 'Mountain',
    'Idaho': 'Mountain',
    'Montana': 'Mountain',
    'Nevada': 'Mountain',
    'New Mexico': 'Mountain',
    'Utah': 'Mountain',
    'Wyoming': 'Mountain',
    'California': 'Pacific',
    'Alaska': 'Pacific',
    'Oregon': 'Pacific',
    'Washington': 'Pacific',
    'Hawaii': 'Pacific'
}

In [634]:
candinfo['census_region'] = candinfo['state'].map(state_to_census_region)

In [635]:
candinfo['year'] = np.full(candinfo.shape[0], 2026)

In [636]:
candinfo.head(3)

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,...,pvi,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",1264611.09,...,-6.455302,-6.714079,0,1,0.0,0.0,0.0,0.0,Pacific,2026
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",83967.63,...,-17.437141,-6.714079,0,0,0.0,0.0,0.0,0.0,East South Central,2026
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2,"FIGURES, SHOMARI C.","HARRIS, HAMPTON",490903.40,...,-6.521708,-6.714079,1,0,0.0,0.0,0.0,0.0,East South Central,2026


In [637]:
## Political scandals
# List of scandals by federal and state officeholders courtesy of Nathaniel Rakich
scandals = pd.read_csv('https://docs.google.com/spreadsheets/d/1ksBLxRR3GCZd33IvhkcNqqBd5K8HwlWC7YuAkVmS1lg/export?format=csv')
scandals.to_csv('../../2026_data/rakich_scandals.csv')
scandals = scandals.set_axis(['politician', 'state', 'position', 'year', 'type', 'outcome', 'comeback'], axis=1)
scandals['politician'] = scandals['politician'].map(lambda x: unicodedata.normalize('NFC', x))
scandals.head()

,politician,state,position,year,type,outcome,comeback
0,Liz Murrill,Louisiana,Attorney General,2026,Intimidation,TBD,NaN
1,Jim Costa,California,Representative,2026,Sexual harassment,TBD,NaN
2,Chuck Edwards,North Carolina,Representative,2026,Sexual harassment,TBD,NaN
3,Eric Swalwell,California,Representative,2026,Sexual assault (rape),Resigned,NaN
4,Sylvia Luke,Hawaii,Lieutenant Governor,2026,Bribery,Retired,NaN


In [638]:
def get_scandals(politician, state, cycle):
    # :param cycle: The year of the election cycle
    mask = ((scandals['state'] == state) & (scandals['year'] <= cycle))
    scan_df = scandals[mask]
    fuzzymatch = process.extractOne(politician, scan_df['politician'], scorer=fuzz.WRatio, score_cutoff=90)
    if fuzzymatch is None:
        return 0
    scan_df_pol = scan_df[scan_df['politician'] == fuzzymatch[0]]
    # Exponential decay factor to put less weight on older scandals relative to newer scandals
    # e^(\delta_t/8)
    scan_df_pol['decay_factor'] = np.exp(-(scan_df_pol['year'].astype(int).map(lambda x: cycle - x))/8)
    pol_scandal_score = scan_df_pol.groupby(['politician']).agg({'decay_factor': 'sum'})['decay_factor'].values[0]
    return pol_scandal_score

In [639]:
candinfo['dem_scandal_score'] = candinfo.apply(lambda x: get_scandals(x['dem_cand'], x['state'], x['year']), axis=1)
candinfo['rep_scandal_score'] = candinfo.apply(lambda x: get_scandals(x['rep_cand'], x['state'], x['year']), axis=1)
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,...,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",1264611.09,...,0,1,0.0,0.0,0.0,0.0,Pacific,2026,0.0,0.0
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",83967.63,...,0,0,0.0,0.0,0.0,0.0,East South Central,2026,0.0,0.0
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2,"FIGURES, SHOMARI C.","HARRIS, HAMPTON",490903.40,...,1,0,0.0,0.0,0.0,0.0,East South Central,2026,0.0,0.0
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL",60787.05,...,0,1,0.0,0.0,0.0,0.0,East South Central,2026,0.0,0.0
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP.",15462.00,...,0,1,0.0,0.0,0.0,0.0,East South Central,2026,0.0,0.0


In [640]:
# Dictionary courtesy of Zach Williams on ActiveState: https://code.activestate.com/recipes/577775-state-fips-codes-dict/
state_codes = {
    'WA': '53', 'DE': '10', 'DC': '11', 'WI': '55', 'WV': '54', 'HI': '15',
    'FL': '12', 'WY': '56', 'PR': '72', 'NJ': '34', 'NM': '35', 'TX': '48',
    'LA': '22', 'NC': '37', 'ND': '38', 'NE': '31', 'TN': '47', 'NY': '36',
    'PA': '42', 'AK': '02', 'NV': '32', 'NH': '33', 'VA': '51', 'CO': '08',
    'CA': '06', 'AL': '01', 'AR': '05', 'VT': '50', 'IL': '17', 'GA': '13',
    'IN': '18', 'IA': '19', 'MA': '25', 'AZ': '04', 'ID': '16', 'CT': '09',
    'ME': '23', 'MD': '24', 'OK': '40', 'OH': '39', 'UT': '49', 'MO': '29',
    'MN': '27', 'MI': '26', 'RI': '44', 'KS': '20', 'MT': '30', 'MS': '28',
    'SC': '45', 'KY': '21', 'OR': '41', 'SD': '46'
}

In [641]:
candinfo['fips'] = candinfo['state_po'].map(state_codes)
candinfo['geoid'] = candinfo['fips'] + candinfo['district_number'].map(lambda x: f'0{x}' if x < 10 else str(x))
candinfo.head(2)

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,...,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,fips,geoid
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",1264611.09,...,0.0,0.0,0.0,0.0,Pacific,2026,0.0,0.0,02,0200
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",83967.63,...,0.0,0.0,0.0,0.0,East South Central,2026,0.0,0.0,01,0101


In [642]:
clust_model = joblib.load('../../model/demographic_kmeans.pkl')
candinfo['demo_cluster'] = clust_model.predict(candinfo[['cvap_white_pct', 'cvap_aapi_pct', 'cvap_black_pct', 'cvap_hisp_pct', 'cvap_natam_pct', 'college']])

In [643]:
candinfo.head(3)

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,...,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,fips,geoid,demo_cluster
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",1264611.09,...,0.0,0.0,0.0,Pacific,2026,0.0,0.0,02,0200,0
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",83967.63,...,0.0,0.0,0.0,East South Central,2026,0.0,0.0,01,0101,0
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2,"FIGURES, SHOMARI C.","HARRIS, HAMPTON",490903.40,...,0.0,0.0,0.0,East South Central,2026,0.0,0.0,01,0102,4


In [644]:
for party in ['dem', 'rep']:
    candinfo[f'{party}_poll_avg'] = candinfo[f'{party}_poll_avg'].fillna(0)

In [645]:
candinfo.to_csv('transformed/2026_house_prediction_dataset.csv')